In [1]:
    # Cell 1: Load API Key

import os
from dotenv import load_dotenv

    # Try to load from a local .env file first
load_dotenv()

    # Check if the key is in the environment variables
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    try:
        # If not found locally, try to get it from Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
        os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
        print("✅ Gemini API key loaded from Kaggle Secrets.")
    except Exception:
        print("⚠️ Warning: GOOGLE_API_KEY not found. Please add it to your .env file or Kaggle Secrets.")
else:
    print("✅ Gemini API key loaded from local environment.")

✅ Gemini API key loaded from local environment.


In [2]:
    # Cell 2: Imports and Proxy Helper
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types
from IPython.display import display, HTML

    # ✅ FIX: Make jupyter_server import optional so it doesn't crash locally
try:
    from jupyter_server.serverapp import list_running_servers
    HAS_JUPYTER_SERVER = True
except ImportError:
    HAS_JUPYTER_SERVER = False

def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"
    url_prefix = ""
    url = "http://127.0.0.1:8000" # Default for local machines

    # Only try to detect Kaggle environment if the library is available
    if HAS_JUPYTER_SERVER:
        try:
            servers = list(list_running_servers())
            if servers:
                baseURL = servers[0]["base_url"]
                path_parts = baseURL.split("/")
                if len(path_parts) >= 4:
                    kernel = path_parts[2]
                    token = path_parts[3]
                    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
                    url = f"{PROXY_HOST}{url_prefix}"
        except Exception:
            pass  # If anything fails, fall back to the local default URL

    # Display a nice clickable button for the user
    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <strong>⚠️ IMPORTANT:</strong> The ADK web UI is not running yet. 
        <ol>
            <li>Run the <code>!adk web</code> cell further down.</li>
            <li>Wait for it to say "Running" (it will not finish).</li>
            <li>Then, click the button below to open the UI.</li>
        </ol>
        <a href='{url}' target='_blank' style="display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px; text-decoration: none; border-radius: 25px; font-weight: bold;">
            Open ADK Web UI ↗
        </a>
    </div>
    """
    display(HTML(styled_html))
    return url_prefix

print("✅ Libraries imported and helper function defined.")

✅ Libraries imported and helper function defined.


In [3]:
    # Cell 3: Define the Agent and Runner
    # Set up retry logic to prevent crashes on temporary network errors
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504]
)

# Build the Agent using your confirmed working model
root_agent = Agent(
    name="helpful_assistant",
    model=Gemini(
        model="gemini-3.1-flash-lite",  # ✅ Your confirmed working model!
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if you are unsure.",
    tools=[google_search], # Gives the agent the power to search the live internet
)

# The Runner is the "engine" that executes the agent's logic
runner = InMemoryRunner(agent=root_agent)

print("✅ Agent and Runner created successfully with gemini-3.1-flash-lite.")

✅ Agent and Runner created successfully with gemini-3.1-flash-lite.


In [4]:
# Cell 4: Test the Agent
print("Asking the agent a question...\n")

# We use 'await' because the agent might take a moment to search the web and think
response = await runner.run_debug("What is the Agent Development Kit from Google? What languages is the SDK available in?")

print(response)

Asking the agent a question...

helpful_assistant > The **Agent Development Kit (ADK)** is an open-source framework from Google designed to help developers build, debug, and deploy reliable AI agents and multi-agent systems at an enterprise scale.

It aims to make agent development feel more like standard software development by providing a structured, flexible, and modular foundation. While it is optimized for use with Google’s Gemini models, the ADK is **model-agnostic** (it can integrate with various Large Language Models) and **deployment-agnostic** (agents can run locally, on Google Cloud, or be containerized for other environments).

### Key Features
*   **Multi-Agent Support:** It natively supports complex architectures where multiple specialized agents can collaborate, delegate tasks, and use dynamic routing.
*   **Tool Ecosystem:** It allows agents to integrate with custom functions, APIs, and third-party tools to perform real-world actions.
*   **Local Debugging:** The ADK in

In [5]:
# Cell 5.1: Create the agent files on disk using your working model
!adk create sample_agent --model gemini-3.1-flash-lite --api_key $GOOGLE_API_KEY

^C


In [ ]:
# Cell 5.2: Generate the clickable link (Run this, then read the output box)
url_prefix = get_adk_proxy_url()

In [ ]:
# 1. Change into the sample-agent folder
%cd sample-agent

# 2. Start the web server from inside that folder
!adk web

In [ ]:
# Cell 5.3: Start the Web Server
# ⚠️ IMPORTANT: Run this cell LAST. It will stay "Running". 
# Go back to Cell 5.2 and click the blue button to open the UI in a new tab.
if url_prefix:
    print(f"Starting ADK Web UI with Kaggle proxy...")
    !adk web --url_prefix {url_prefix}
else:
    print("Starting ADK Web UI locally at http://127.0.0.1:8000 ...")
    !adk web

In [ ]:
# THE MASTER DIAGNOSTIC CELL

import os

print("1. Current Directory:", os.getcwd())

# 2. Change to the sample-agent folder
%cd sample-agent

# 3. Check if .env exists and actually has your API key
if os.path.exists(".env"):
    with open(".env", "r") as f:
        env_contents = f.read().strip()
        print("2. ✅ .env file found. Contents:")
        print(env_contents)
        
        # Quick check to make sure the key isn't empty
        if "GOOGLE_API_KEY=" in env_contents and len(env_contents) > 30:
            print("   -> API key looks valid!")
        else:
            print("   -> ⚠️ WARNING: API key looks missing or empty! Please edit the .env file.")
else:
    print("2. ❌ ERROR: .env file NOT found in the sample-agent folder!")

# 4. Start the server on a fresh port (8001) to guarantee no port conflicts
print("\n3. 🚀 Starting ADK Web UI on port 8001...")
print("   (Wait for the 'Uvicorn running' message below)\n")
!adk web --port 800